# ভিশন-ল্যাঙ্গুয়েজ প্রিট্রেইনিং অবজেকটিভ

ভিশন-ল্যাঙ্গুয়েজ pretraining অবজেকটিভ — CLIP-এর softmax InfoNCE, SigLIP-এর pairwise sigmoid loss, এবং একটি generative captioning loss; সবগুলোই synthetic ডেটায় scratch থেকে প্রশিক্ষণ দেওয়া হয়েছে এবং প্রকৃত মাপার সংখ্যা দিয়ে তুলনা করা হয়েছে।

কোনো download নেই, কোনো pretrained weight নেই। কেবল CPU; পুরো script কয়েক মিনিট নেয় কারণ পরীক্ষা 1 আটটি encoder-কে convergence পর্যন্ত প্রশিক্ষণ দেয়।

দুটি পরীক্ষা, দুটি synthetic জগৎ:

- **পরীক্ষা 1 ও 2 (contrastive loss):** প্রতিটি জোড়া একটি continuous latent "concept" ভেক্টর ভাগ করে নেয়; image view আর text view সেটির ভিন্ন noisy linear rendering। Retrieval তাই well posed: প্রতিটি ইমেজের জন্য gallery-তে ঠিক একটিমাত্র caption।
- **পরীক্ষা 3 (একটি অবজেকটিভ কী রেখে দেয়):** প্রতিটি ইমেজ একটি CATEGORY ও একটি ATTRIBUTE-এর object দেখায়, আর দুটোই ইমেজে দৃশ্যমান — কিন্তু caption কেবল category-র নাম বলে, ঠিক যেমন প্রকৃত ওয়েব alt-text "a dog" বলে আর "a small brown dog facing left" বলে না। এরপর দেখা হয় কোন training objective attribute-টিকে frozen feature থেকে পুনরুদ্ধারযোগ্য রাখে।

চালানোর নিয়ম: উপর থেকে নিচ পর্যন্ত সব cell চালান — প্রতিটি অংশের code cell নিজেই তার ডেমো চালায়। (একই টিউটোরিয়ালের পাইথন-স্ক্রিপ্ট সংস্করণ: `example.py`)


In [ ]:
"""ভিশন-ল্যাঙ্গুয়েজ প্রিট্রেইনিং অবজেকটিভ -- CLIP-এর softmax InfoNCE, SigLIP-এর
pairwise sigmoid loss, এবং একটি generative captioning loss; সবগুলোই synthetic
ডেটায় scratch থেকে প্রশিক্ষিত এবং প্রকৃত মাপার সংখ্যা দিয়ে তুলনা করা হয়েছে।

কোনো download নেই, কোনো pretrained weight নেই। কেবল CPU; পুরো script কয়েক মিনিট
নেয় কারণ পরীক্ষা 1 আটটি encoder-কে convergence পর্যন্ত প্রশিক্ষণ দেয়।

দুটি পরীক্ষা, দুটি synthetic জগৎ:

  পরীক্ষা 1 ও 2 (contrastive loss): প্রতিটি জোড়া একটি continuous latent
  "concept" ভেক্টর ভাগ করে নেয়; image view আর text view সেটির ভিন্ন noisy
  linear rendering। Retrieval তাই well posed: প্রতিটি ইমেজের জন্য gallery-তে
  ঠিক একটিমাত্র caption।

  পরীক্ষা 3 (একটি অবজেকটিভ কী রেখে দেয়): প্রতিটি ইমেজ একটি CATEGORY ও একটি
  ATTRIBUTE-এর object দেখায়, আর দুটোই ইমেজে দৃশ্যমান -- কিন্তু caption কেবল
  category-র নাম বলে, ঠিক যেমন প্রকৃত ওয়েব alt-text "a dog" বলে আর
  "a small brown dog facing left" বলে না। আমরা তারপর দেখি কোন training
  objective attribute-টিকে frozen feature থেকে পুনরুদ্ধারযোগ্য রাখে।

চালানোর নিয়ম:
    python example.py
"""

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

LATENT = 10
IMG_RAW = 32
TXT_RAW = 24
EMBED = 16


## 0. দুটি synthetic ডেটা জেনারেটর

পরীক্ষা 1 ও 2-র জন্য `sample_pairs`: একটি shared latent থেকে image/text দুই ভিউ। পরীক্ষা 3-র জন্য `sample_scenes`: ইমেজে category + attribute দুটোই আছে, কিন্তু caption-এ শুধু category — ঠিক যেমন প্রকৃত ওয়েব alt-text। সঙ্গে একটি ক্ষুদ্র MLP `Tower` encoder ও normalization helper।


In [ ]:
# ---------------------------------------------------------------------------
# 0. দুটি synthetic ডেটা জেনারেটর
# ---------------------------------------------------------------------------

img_render = torch.randn(LATENT, IMG_RAW)
txt_render = torch.randn(LATENT, TXT_RAW)


def sample_pairs(n):
    """একটি continuous shared latent থেকে একটি image view ও একটি text view।"""
    z = torch.randn(n, LATENT)
    img = z @ img_render + 0.9 * torch.randn(n, IMG_RAW)
    txt = z @ txt_render + 0.9 * torch.randn(n, TXT_RAW)
    return img, txt


NUM_CATEGORIES = 12
NUM_ATTRIBUTES = 4
cat_latents = torch.randn(NUM_CATEGORIES, LATENT)
attr_latents = torch.randn(NUM_ATTRIBUTES, LATENT)


def sample_scenes(n):
    """ইমেজে category ও attribute দুটোই থাকে; caption-এ কেবল category।"""
    cats = torch.randint(0, NUM_CATEGORIES, (n,))
    attrs = torch.randint(0, NUM_ATTRIBUTES, (n,))
    img = (cat_latents[cats] + attr_latents[attrs]) @ img_render + 0.3 * torch.randn(n, IMG_RAW)
    txt = cat_latents[cats] @ txt_render + 0.05 * torch.randn(n, TXT_RAW)
    return img, txt, cats, attrs


class Tower(nn.Module):
    """একটি vision বা text encoder-এর ক্ষুদ্র প্রতিস্থাপক।"""

    def __init__(self, in_dim, out_dim=EMBED):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.GELU(),
            nn.Linear(64, 64), nn.GELU(),
            nn.Linear(64, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def normalized(x):
    return F.normalize(x, dim=-1)


## 1. দুটি contrastive loss

InfoNCE (CLIP): batch-জুড়ে softmax cross-entropy, উভয় দিকে। SigLIP: প্রতিটি জোড়ায় independent sigmoid, batch-জুড়ে কোনো normalization ছাড়া। এরপর একই উদাহরণ সংখ্যা দিয়ে (শুধু batch size ভিন্ন) আটটি encoder প্রশিক্ষণ দিয়ে held-out 256-way retrieval-এ R@1 মাপা হয় — দেখা যাবে batch বাড়লে InfoNCE কতটা দ্রুত ওঠে, আর sigmoid loss ছোট batch-এও কতটা সহনশীল।


In [ ]:
# ---------------------------------------------------------------------------
# 1. দুটি contrastive loss
# ---------------------------------------------------------------------------

def infonce_loss(img_e, txt_e, logit_scale):
    """CLIP: batch-জুড়ে softmax cross-entropy, উভয় দিকে, গড় নেওয়া।

    batch-এর প্রতিটি অন্য আইটেমই negative এবং softmax batch-এর across
    normalize করে, তাই কাজের কঠিনতা -- এবং একটি gradient step বহন করতে পারে
    তথ্যের পরিমাণ, সর্বোচ্চ log(N) nats -- সরাসরি batch size-এর সাথে যুক্ত।
    """
    logits = logit_scale.exp() * img_e @ txt_e.t()          # (N, N)
    labels = torch.arange(img_e.shape[0])
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))


def siglip_loss(img_e, txt_e, logit_scale, logit_bias):
    """SigLIP: প্রতিটি জোড়ায় independent binary classification।

    N*N জোড়ার প্রতিটির নিজস্ব sigmoid -- মিল থাকা জোড়া positive, বাকি সব
    negative। batch-জুড়ে কিছুই normalize হয় না, তাই ছোট N-এও loss well posed
    থাকে। এটি একটি learned BIAS চায় কারণ জোড়ার matrix প্রধানত negative
    (N positive বনাম N^2 - N negative)।
    """
    n = img_e.shape[0]
    logits = logit_scale.exp() * img_e @ txt_e.t() + logit_bias
    targets = 2 * torch.eye(n) - 1                          # ডায়াগোনালে +1, অন্যত্র -1
    return -F.logsigmoid(targets * logits).sum() / n


@torch.no_grad()
def retrieval_r1(img_tower, txt_tower, gallery=256, trials=12):
    """`gallery`-way gallery-তে held-out image->text retrieval R@1।"""
    hits = 0
    for _ in range(trials):
        img, txt = sample_pairs(gallery)
        ie, te = normalized(img_tower(img)), normalized(txt_tower(txt))
        hits += ((ie @ te.t()).argmax(dim=1) == torch.arange(gallery)).sum().item()
    return hits / (trials * gallery)


TOTAL_EXAMPLES = 32_000        # batch size জুড়ে অপরিবর্তিত -- নিচে দেখুন


def train_contrastive(objective, batch_size, lr=3e-3):
    torch.manual_seed(1)                                    # সব জায়গায় অভিন্ন init
    img_tower, txt_tower = Tower(IMG_RAW), Tower(TXT_RAW)
    logit_scale = nn.Parameter(torch.tensor(math.log(10.0)))
    logit_bias = nn.Parameter(torch.tensor(-10.0))
    params = list(img_tower.parameters()) + list(txt_tower.parameters()) + [logit_scale, logit_bias]
    opt = torch.optim.Adam(params, lr=lr)

    steps = TOTAL_EXAMPLES // batch_size                    # সমান DATA, সমান step নয়
    for _ in range(steps):
        img, txt = sample_pairs(batch_size)
        ie, te = normalized(img_tower(img)), normalized(txt_tower(txt))
        if objective == "infonce":
            loss = infonce_loss(ie, te, logit_scale)
        else:
            loss = siglip_loss(ie, te, logit_scale, logit_bias)
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            logit_scale.clamp_(0, math.log(100.0))          # CLIP temperature-কে clamp করে
    return img_tower, txt_tower, logit_scale, logit_bias, steps


print("=" * 78)
print("1. InfoNCE (CLIP) vs SIGMOID (SigLIP) ACROSS BATCH SIZES")
print("=" * 78)
print(f"Every run sees the SAME {TOTAL_EXAMPLES:,} training examples (so a small batch")
print("simply takes more steps) and starts from the same initialization.")
print("Score: held-out image->text retrieval R@1 in 256-way galleries (chance 0.4%).")
print()
print(f"{'batch':>7} {'steps':>7} {'negatives/anchor':>18} {'InfoNCE R@1':>13} {'Sigmoid R@1':>13}")

results = {}
for bs in [4, 16, 64, 256]:
    nce = train_contrastive("infonce", bs)
    sig = train_contrastive("sigmoid", bs)
    r_nce, r_sig = retrieval_r1(nce[0], nce[1]), retrieval_r1(sig[0], sig[1])
    results[bs] = (nce, sig, r_nce, r_sig)
    print(f"{bs:>7} {nce[4]:>7} {bs - 1:>18} {r_nce:>12.1%} {r_sig:>12.1%}")

print()
print("Both losses pull matched pairs together and push mismatched pairs apart;")
print("they differ in HOW negatives enter. InfoNCE's softmax is normalized across")
print("the batch, so with batch 4 the model only ever has to beat 3 distractors --")
print("it solves that easy task and stops improving, then transfers poorly to a")
print("256-way gallery. Give it more negatives and it keeps getting better. The")
print("sigmoid loss scores each pair independently against an absolute threshold,")
print("so it degrades far more gently as the batch shrinks.")
print()
print("This is why CLIP-style training needs enormous batches -- 32,768 in the")
print("original paper, sharded across hundreds of devices with an all-gather so")
print("that every device's softmax sees the global batch -- and why SigLIP was")
print("proposed: no all-gather, no global softmax, competitive quality at batch")
print("sizes that fit on far less hardware.")
print()
print(f"the logits matrix at CLIP's real batch size: {32768 ** 2:,} cells")
print(f"                                   in fp16: {32768 ** 2 * 2 / 1e9:.1f} GB for ONE matrix,")
print("which is why large-batch InfoNCE is implemented in chunks rather than")
print("materialized in full.")
print()


## 2. Learned temperature ও bias

Cosine similarity `[−1, 1]`-এ সীমাবদ্ধ; সেটি সরাসরি softmax-এ গেলে encoder যত ভালোই হোক প্রায় uniform বণ্টন ও ক্ষীণ gradient পাওয়া যায়। তাই temperature একটি learned parameter, আর SigLIP-এর bias negative দিকে থাকে কারণ জোড়ার বিপুলাংশই negative। এখানে দুটোই কোথায় গিয়ে থামে তা দেখা হয়।


In [ ]:
# ---------------------------------------------------------------------------
# 2. Learned temperature ও bias
# ---------------------------------------------------------------------------

print("=" * 78)
print("2. WHAT THE LEARNED TEMPERATURE AND BIAS CONVERGE TO")
print("=" * 78)
for bs in [4, 64, 256]:
    nce, sig, _, _ = results[bs]
    print(f"batch {bs:>4}:  InfoNCE exp(logit_scale) = {nce[2].exp().item():6.1f}"
          f"   |   SigLIP exp(logit_scale) = {sig[2].exp().item():6.1f}, bias = {sig[3].item():7.2f}")
print()
print("Cosine similarities live in [-1, 1]. Feed those straight into a softmax and")
print("the distribution is nearly uniform no matter how good the encoders are, so")
print("gradients stay tiny; the temperature is therefore a LEARNED parameter that")
print("grows until the similarity distribution is sharp enough to be informative.")
print("(CLIP clamps it at 100 to keep training stable -- so does this script.)")
print("SigLIP's extra learned bias sits strongly negative because N^2 - N of the")
print("N^2 pairs are negatives: without it, 'no match anywhere' is an excellent")
print("local minimum for the first few thousand steps.")
print()


## 3. কোন অবজেকটিভ এনকোডারকে কী রাখতে বাধ্য করে

প্রতিটি synthetic ইমেজে একটি category ও একটি attribute আছে, কিন্তু caption কেবল category বলে। তিনভাবে প্রশিক্ষণ দিয়ে (contrastive, captioning-শুধু category, captioning- category + attribute) প্রতিটি image tower freeze করে তার feature-এ linear probe বসানো হয়, সঙ্গে `collapse_ratio` — একটি category-র ইমেজগুলো কতটা এক বিন্দুতে চেপে ফেলা হয়েছে তার অনুপাত।


In [ ]:
# ---------------------------------------------------------------------------
# 3. কোন অবজেকটিভ এনকোডারকে কী রাখতে বাধ্য করে
# ---------------------------------------------------------------------------

def train_contrastive_scenes(steps=3000, bs=64, lr=3e-3):
    """Contrastive training যেখানে caption কেবল category-র নাম বলে।"""
    torch.manual_seed(1)
    img_tower, txt_tower = Tower(IMG_RAW), Tower(TXT_RAW)
    logit_scale = nn.Parameter(torch.tensor(math.log(10.0)))
    opt = torch.optim.Adam(list(img_tower.parameters()) + list(txt_tower.parameters())
                           + [logit_scale], lr=lr)
    for _ in range(steps):
        img, txt, _, _ = sample_scenes(bs)
        ie, te = normalized(img_tower(img)), normalized(txt_tower(txt))
        loss = infonce_loss(ie, te, logit_scale)
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            logit_scale.clamp_(0, math.log(100.0))
    return img_tower


def train_captioning(steps=3000, bs=64, lr=3e-3, caption_has_attribute=False):
    """Generative supervision: ইমেজ থেকে caption-এর content পূর্বাভাস করা।

    প্রকৃত captioner হল caption জুড়ে autoregressive next-token prediction
    (Phase 01 / Phase 04-এর LM loss)। এখানে শুধু এটাই গুরুত্বপূর্ণ যে
    supervision প্রতি content-token-এ আসে, প্রতি জোড়ায় একটি similarity score
    নয়।
    """
    torch.manual_seed(1)
    img_tower = Tower(IMG_RAW)
    cat_head = nn.Linear(EMBED, NUM_CATEGORIES)
    attr_head = nn.Linear(EMBED, NUM_ATTRIBUTES)
    opt = torch.optim.Adam(list(img_tower.parameters()) + list(cat_head.parameters())
                           + list(attr_head.parameters()), lr=lr)
    for _ in range(steps):
        img, _, cats, attrs = sample_scenes(bs)
        feats = img_tower(img)
        loss = F.cross_entropy(cat_head(feats), cats)
        if caption_has_attribute:
            loss = loss + F.cross_entropy(attr_head(feats), attrs)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return img_tower


@torch.no_grad()
def frozen_feats(tower, n):
    img, _, cats, attrs = sample_scenes(n)
    return normalized(tower(img)), cats, attrs


def linear_probe(tower, target, train_n=4000, test_n=2000, steps=800):
    """Encoder freeze করে তার feature-এ একটি linear classifier বসিয়ে test
    accuracy মাপা: তথ্যটি কি এখনও representation-এ linearly আছে?"""
    n_classes = NUM_ATTRIBUTES if target == "attr" else NUM_CATEGORIES
    Xtr, ctr, atr = frozen_feats(tower, train_n)
    Xte, cte, ate = frozen_feats(tower, test_n)
    ytr, yte = (atr, ate) if target == "attr" else (ctr, cte)
    probe = nn.Linear(EMBED, n_classes)
    opt = torch.optim.Adam(probe.parameters(), lr=0.05)
    for _ in range(steps):
        loss = F.cross_entropy(probe(Xtr), ytr)
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        return (probe(Xte).argmax(1) == yte).float().mean().item()


@torch.no_grad()
def collapse_ratio(tower, n=4000):
    """Within-category spread ভাগ by between-category spread।

    ছোট হলে বোঝায় যে একই caption ভাগ করা সব ইমেজ এক বিন্দুতে চেপে ফেলা হয়েছে,
    তাই সেই ইমেজগুলোর মধ্যে যা পার্থক্য ছিল তা চলে গেছে।
    """
    X, cats, _ = frozen_feats(tower, n)
    within = torch.stack([X[cats == k].var(0).sum() for k in range(NUM_CATEGORIES)
                          if (cats == k).sum() > 1]).mean()
    between = torch.stack([X[cats == k].mean(0) for k in range(NUM_CATEGORIES)]).var(0).sum()
    return (within / between).item()


print("=" * 78)
print("3. WHAT EACH OBJECTIVE FORCES THE ENCODER TO KEEP")
print("=" * 78)
print("Every image shows a CATEGORY and an ATTRIBUTE. The captions name the")
print("category only -- the attribute is visible in the pixels and absent from")
print("the text, exactly like real web alt-text. We freeze each trained image")
print("tower and linear-probe its features for both facts.")
print()

rows = [
    ("contrastive (InfoNCE)", train_contrastive_scenes()),
    ("captioning, category only", train_captioning(caption_has_attribute=False)),
    ("captioning, category + attribute", train_captioning(caption_has_attribute=True)),
]
print(f"{'image tower trained with':<34} {'category':>9} {'attribute':>10} {'within/between':>15}")
for name, tower in rows:
    print(f"{name:<34} {linear_probe(tower, 'cat'):>8.1%} {linear_probe(tower, 'attr'):>9.1%}"
          f" {collapse_ratio(tower):>15.4f}")
print(f"{'chance':<34} {1 / NUM_CATEGORIES:>8.1%} {1 / NUM_ATTRIBUTES:>9.1%}")
print()
print("Every tower nails the category -- that is what all three objectives are")
print("supervised on. They differ on the attribute, which no caption mentions.")
print()
print("The last column explains why. Contrastive training's target for an image")
print("is its caption embedding, and here all images of a category share one")
print("caption, so the loss actively squashes them onto the same point: its")
print("within-category spread is several times smaller than the captioner's, and")
print("the attribute -- the thing that distinguished those images -- degrades")
print("with it. The captioner is only ever asked to predict the category and")
print("still preserves the attribute far better, because a generative head has")
print("no incentive to actively DESTROY information it doesn't use.")
print()
print("This is the practical argument against building a VLM on a contrastive")
print("tower alone: a representation optimized to match short web captions is")
print("optimized to discard everything those captions omit, which is exactly the")
print("fine detail -- counts, small text, spatial relations, attributes -- a VLM")
print("gets asked about later. Real pipelines respond by mixing objectives")
print("(CoCa-style contrastive + captioning), by training on dense re-captions")
print("rather than raw alt-text, or by fusing a contrastive tower with a")
print("self-supervised one such as DINOv2 (Lesson 1 section 2).")
